# Whole-file vs. function-level LM-CC

Side-by-side comparison of LM-CC scoped to the whole file vs. the patched function.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.config import PATHS
import pandas as pd

file_df = pd.read_parquet(PATHS.WHOLE_FILE_RESULT_DATASET)
function_df = pd.read_parquet(PATHS.FUNCTION_RESULT_DATASET)
difficulty = pd.read_parquet(PATHS.TASK_DIFFICULTY)

merged = (
    file_df.merge(function_df, on='instance_id', suffixes=('_file', '_fn'))
    .merge(difficulty[['instance_id', 'resolution_rate']], on='instance_id', how='left')
)
clean = merged[
    merged['has_patched_function'].fillna(False)
    & merged['parsable_file'].fillna(False)
    & merged['parsable_fn'].fillna(False)
    & merged['lm_cc_score_file'].notna()
    & merged['lm_cc_score_fn'].notna()
    & merged['resolution_rate'].notna()
].copy()
print(f'Tasks compared (file & function both available): {len(clean)}')


Tasks compared (file & function both available): 411


## 1. Collinearity with LOC

In [2]:
from scipy.stats import spearmanr
rho_file, _ = spearmanr(clean['lm_cc_score_file'], clean['loc_file'])
rho_fn, _   = spearmanr(clean['lm_cc_score_fn'], clean['loc_fn'])
print('rho(LM-CC, LOC):')
print(f'  whole file : {rho_file:+.3f}')
print(f'  function   : {rho_fn:+.3f}')


rho(LM-CC, LOC):
  whole file : +0.985
  function   : +0.986


## 2. LM-CC sample partial controlling for LOC (vs. resolution rate)

In [3]:
from src.correlation import partial_spearman
pf_file, _ = partial_spearman(clean, 'lm_cc_score_file', 'resolution_rate', 'loc_file')
pf_fn, _   = partial_spearman(clean, 'lm_cc_score_fn', 'resolution_rate', 'loc_fn')
print('LM-CC partial(resolution_rate | LOC):')
print(f'  whole file : {pf_file:+.3f}')
print(f'  function   : {pf_fn:+.3f}')


LM-CC partial(resolution_rate | LOC):
  whole file : -0.150
  function   : +0.016


## 3. Full correlation tables side by side

In [4]:
from src.correlation import full_correlation_table
file_metrics = ['lm_cc_score_file', 'cc_sum_file', 'nesting_max_file']
fn_metrics   = ['lm_cc_score_fn', 'cc_sum_fn', 'nesting_max_fn']
print('=== WHOLE FILE (control = loc_file) ===')
print(full_correlation_table(clean, file_metrics, score='resolution_rate', control='loc_file').to_string(index=False))
print()
print('=== FUNCTION (control = loc_fn) ===')
print(full_correlation_table(clean, fn_metrics, score='resolution_rate', control='loc_fn').to_string(index=False))


=== WHOLE FILE (control = loc_file) ===
          metric      sample_zero sample_partial_loc_file          subgroup_zero subgroup_partial_loc_file
lm_cc_score_file -0.180 (p=0.000)        -0.150 (p=0.002) -0.855 (p=0.002, n=10)    -0.855 (p=0.003, n=10)
     cc_sum_file -0.165 (p=0.001)        -0.058 (p=0.241)  -0.783 (p=0.013, n=9)     -0.783 (p=0.021, n=9)
nesting_max_file -0.192 (p=0.000)        -0.128 (p=0.010)  -0.874 (p=0.002, n=9)                         —

=== FUNCTION (control = loc_fn) ===
        metric      sample_zero sample_partial_loc_fn subgroup_zero subgroup_partial_loc_fn
lm_cc_score_fn -0.100 (p=0.043)      +0.016 (p=0.743)             —                       —
     cc_sum_fn -0.076 (p=0.125)      +0.022 (p=0.652)             —                       —
nesting_max_fn -0.046 (p=0.351)      +0.041 (p=0.410)             —                       —
